# Vector Search with Azure Cosmos DB for NoSQL

Vector search is a modern technique for finding similar items or data points in large collections by representing them as vectors. Unlike traditional keyword searches, which look for exact word matches, vector search uses mathematical vectors to capture the meaning and context of data. These vectors are numerical representations that encode various features or attributes of the data.

Vector search, with its efficient comparison of query vectors with data point vectors, ensures the retrieval of the most similar items. This approach, based on mathematical distance, guarantees accurate and relevant search results, particularly in complex and unstructured data applications like images, documents, or natural language queries.

Here’s how it works:

1. **Vectorization**: Each piece of text is transformed into a vector. This vector captures the semantic meaning of the text, allowing for more nuanced comparisons than simple keyword matching.

2. **Indexing**: These vectors are stored in a specialized index that supports efficient similarity searches. 

3. **Querying**: When a search query is made, it is converted into a vector. The system then compares this query vector with the vectors in the index, calculating the distance (e.g., cosine similarity, Euclidean distance) between them.

4. **Retrieval**: The vectors with the smallest distances to the query vector are retrieved, representing the most similar items in the dataset.

Vector search, with its power to handle unstructured data, opens up a world of possibilities. It can be applied to a variety of use cases, from recommendation systems to image retrieval and natural language processing tasks.

## Enable vector search in Azure Cosmos DB for NoSQL

The Vector Search feature in Azure Cosmos DB for NoSQL is currently in preview, so you must first enable the capability. This procedure can be done via the [Azure portal](https://portal.azure.com) or the Azure CLI. Choose whichever method you are most comfortable with and follow the steps for that technique below.

> **Note**: Regardless of the method used, enabling the feature may take several minutes.

###  To enable the feature in the Azure portal:

1. Navigate to your Azure Cosmos DB for NoSQL resource in the [Azure portal](https://portal.azure.com).
2. Expand the **Settings** item in the left-hand menu, select **Features**, and on the **Features** page, select **Vector Search for NoSQL API**.

    ![](https://github.com/solliancenet/azure-data-engineering-conference-workshop-students/blob/master/media/azure-cosmosdb-features-vector-search.png?raw=true)

3. In the **Vector Search for NoSQL API** dialog, review the feature description and select **Enable**.

    ![](https://github.com/solliancenet/azure-data-engineering-conference-workshop-students/blob/master/media/azure-cosmosdb-features-vector-search-enable.png?raw=true)

4. Wait for the notification that the feature was successfully enabled. You should see the **Status** of the **Vector Search for NoSQL API** change to **On**.

### To enable Vector Search via the Azure CLI:

1. From the toolbar in the [Azure portal](https://portal.azure.com), open the Azure Cloud Shell.

    ![](https://github.com/solliancenet/azure-data-engineering-conference-workshop-students/blob/master/media/azure-portal-toolbar-cloud-shell.png?raw=true)

2. At the cloud shell prompt, execute the following command. Ensure you replace the `<resource-group-name>` and Cosmos DB `<account-name>` tokens with the appropriate values from your resource group in Azure.

    ```azurecli
    az cosmosdb update \
        --resource-group <resource-group-name> \
        --name <account-name> \
        --capabilities EnableNoSQLVectorSearch
    ```

3. Wait for the command to run successfully before leaving the Azure Cloud Shell.

## Assign container vector policy and vector indexing policy

Azure Cosmos DB for NoSQL offers several vector indexing options to support efficient vector searching:

- **Flat Index**: This is an exact, brute-force approach to vector indexing. It ensures precise results by comparing each vector in the dataset directly with the query vector.

- **Quantized Flat Index**: Similar to the flat index, but with an added step of quantizing (compressing) the vectors before indexing. This helps in reducing storage requirements while maintaining accuracy.

- **DiskANN Index**: This index uses the DiskANN (Disk-based Approximate Nearest Neighbor) algorithm, which is designed for large-scale, high-dimensional vector data. It enables approximate nearest neighbor search, balancing between accuracy and performance, and is optimized to reduce RU (Request Unit) costs and latency.

**Note**: The Vector Search for NoSQL feature is is still in preview. Therefore, you must define the container vector policy and the vector indexing policy at the time of container creation and cannot apply them to existing containers.

1. In the [Azure portal](https://portal.azure.com), navigate to your Cosmos DB resource.
2. Select **Data Explorer** in the left-hand menu.
3. On the **Data Explorer** page, select **New Container**.

    ![](https://github.com/solliancenet/azure-data-engineering-conference-workshop-students/blob/master/media/azure-cosmos-db-new-container.png?raw=true)

4. In the **New Container** dialog:
    - Select **Create new** under **Database id** and enter "ContosoSuites" as the database name.
    - Uncheck the **Share throughput across containers** box. If checked, this will prevent the creation of a container vector policy.
    - Enter "MaintenanceRequests" into the **Container id** box.
    - Enter "/hotel_id" into the **Partition key** box.
    - Change the **Container Max RU/s** to 1000.
    - Expand the **Container Vectory Policy** section of the dialog, select **Add vector embedding**, and then enter the following values into the specified fields:
      - Path: Enter **"/request_vector"**.
      - Data type: Select **float32**.
      - Distance function: Select **cosine**.
      - Dimensions: Enter **1536**. This is based on the number of dimensions generated by the `ada-text-embedding-002` model in Azure OpenAI.
      - Index type: Select **quantizedFlat**. Given the number of dimensions being specified, 1536, the `flat` index type will not work, as it only supports a maximum of 505 dimensions for vectors. The `diskANN` index could also be used here, but is currently in a limited preview that requires going through a request and approval process.
    - Select **OK** to create the container.

## Populate the database with sample data

For demonstration purposes, we are going to use a small JSON file containing a handful of hotel maintenance requests as our sample data. This file can be found in the `data` folder of the repo you cloned for this lab. To populate the `MaintenanceRequests` container with data from the `PropertyMaintenance.json` file, you will use the Data Explorer for your Azure Cosmos DB account.

1. In the Cosmos DB Data Explorer, expand the **ContosoSuites** database and the **MaintenanceRequests** container, then select **Items**.

  ![](https://github.com/solliancenet/azure-data-engineering-conference-workshop-students/blob/master/media/azure-cosmos-db-data-explorer-maintenance-requests-items.png?raw=true)

2. Select **Upload Item** on the toolbar.

  ![](https://github.com/solliancenet/azure-data-engineering-conference-workshop-students/blob/master/media/azure-cosmos-db-toolbar-upload-item.png?raw=true)

3. In the **Upload Items** dialog, select the browse button and navigate to the `PropertyMaintenance.json` file in the `src\data` directory in the location where cloned the repository, then select **Upload** to import the data in the file.

  ![](https://github.com/solliancenet/azure-data-engineering-conference-workshop-students/blob/master/media/upload-items-property-maintenance.png?raw=true)

    The upload status should indicate 8 documents created.

4. Close the upload dialog and select the refresh icon on the MaintenanceRequests>Items tab in the Data Explorer to view the newly added documents.

    ![](https://github.com/solliancenet/azure-data-engineering-conference-workshop-students/blob/master/media/azure-cosmos-db-maintenance-requests-items-refresh.png?raw=true)

## Install required libraries

You are now ready to start setting up the notebook environment to execute code to create embeddings using Azure OpenAI and execute vector searches with Azure Cosmos DB for NoSQL.

1. Run the following cell to install the required Python libraries.

In [ ]:
%pip install openai==1.42.0
%pip install azure-cosmos==4.7.0
%pip install tabulate

2. Run the next cell to import the required libraries and objects into the environment.

In [ ]:
from azure.cosmos import CosmosClient
import openai
from tabulate import tabulate

## Create a function to call Azure OpenAI to generate embeddings

In [ ]:
def generate_embeddings(text: str) -> list:
    """Create and return a new embedding request. Key assumptions:
    - Azure OpenAI endpoint, key, and deployment name stored in Streamlit secrets."""

    aoai_endpoint = ""
    aoai_key = ""
    aoai_embedding_deployment_name = "text-embedding-ada-002"

    client = openai.AzureOpenAI(
        api_key=aoai_key,
        api_version="2024-06-01",
        azure_endpoint = aoai_endpoint
    )
    # Execute an embedding request
    response = client.embeddings.create(
        model = aoai_embedding_deployment_name,
        input = text
    )
    
    return response.data[0].embedding

In [ ]:
# Test the function and observe the output.
generate_embeddings("Hello, world!")

## Generate embeddings for existing records

For demonstration purposes, a simple function call is being used to add vector embeddings to the records in the database. In a real-world scenario, the typical approach is to leverage the Cosmos DB change feed, and use either an Azure Function with a Cosmos DB trigger or a worker process in a container app to process the change feed.

In [ ]:
# Define the Cosmos DB connection parameters
cosmos_endpoint = ""
cosmos_key = ""
cosmos_database_name = "ContosoSuites"
cosmos_container_name = "MaintenanceRequests"

In [ ]:
def vectorize_data():
    """Query the Cosmos DB, create embeddings for the details field, and then insert them back into Cosmos DB."""
        
    # Create a Cosmos DB client
    client = CosmosClient(cosmos_endpoint, cosmos_key)
    database = client.get_database_client(cosmos_database_name)
    container = database.get_container_client(cosmos_container_name)
    
    # Query the Cosmos DB for maintenance requests
    query = "SELECT * FROM c"
    items = list(container.query_items(query=query, enable_cross_partition_query=True))
    
    # Create embeddings for the details field
    for item in items:
        # Combine the hotel and details fields into a single string for embedding.
        text_to_embed = f'Hotel: {item["hotel"]}\nRequest Details: {item["details"]}'
        item["request_embedding"] = generate_embeddings(text_to_embed)
        container.upsert_item(item)

    print(f"Embeddings created for {len(items)} maintenance requests.")

In [ ]:
vectorize_data()

## Review vectorized data in Cosmos DB

1. Return to the [Azure portal](https://portal.azure.com/) and navigate to your Cosmos DB account.
2. Open the `MaintenanceRequests` items collection.
3. Select a few random request items to ensure they contain a populated `request_vector` value.

## Execute vector search queries

Vector search is quite powerful for textual data. It works by taking each piece of text, whether it’s a sentence, paragraph, or an entire document, and converting into a vector that captures its semantic meaning. For example, if you have a collection of news articles and you want to find articles similar to one about climate change, the system will convert the text of each article into vectors. When you input your climate change article as a query, it gets converted into a vector as well. The system then compares this query vector with the vectors of all the articles in the collection, identifying those that are most similar based on their mathematical distance. This allows you to find articles that are contextually and semantically similar, even if they don’t share the same keywords.

1. Define a function to execute a vector search against Azure Cosmos DB. The query uses the `VectorDistance()` function to perform a cosine distance similarity search between the stored vector and the vectorized seach query text.

In [ ]:
def execute_vector_search(query_embedding, max_results=5, minimum_similarity_score=0.8):
    """Execute a vector search query."""

    # Create a CosmosClient
    client = CosmosClient(url=cosmos_endpoint, credential=cosmos_key)
    # Load the Cosmos database and container
    database = client.get_database_client(cosmos_database_name)
    container = database.get_container_client(cosmos_container_name)

    query = f"""
            SELECT TOP {max_results}
                c.hotel_id,
                c.hotel,
                c.details,
                VectorDistance(c.request_vector, @request_vector) AS SimilarityScore
            FROM c
            WHERE
                VectorDistance(c.request_vector, @request_vector) > {minimum_similarity_score}
            ORDER BY
                VectorDistance(c.request_vector, @request_vector)
            """

    results = container.query_items(
        query=query,
        parameters=[
            {"name": "@request_vector", "value": query_embedding}
        ],
        enable_cross_partition_query=True
    )

    # Create and return a new vector search request
    return results

2. Execute a query using natural language, such as "air conditioning is not working".

In [ ]:
embeddings = generate_embeddings("air conditioning is not working")
results = execute_vector_search(embeddings)

data = []
for item in results:
    data.append([item["hotel_id"], item["hotel"], item["details"], item["SimilarityScore"]])

headers = ["Hotel Id", "Hotel", "Details", "Similarity Score"]

# Create a table to display the results
table = tabulate(data, headers, tablefmt="grid")

print(table)

3. Now, execute another query, this time looking for requests related to fire and safety issues.

    Note, the call to `execute_vector_search()` now includes values for `max_results` and `minimum_similarity_score`. These allow you to limit the number of results both in number and by relevance. By allowing more results and a lower similarity score threshold, you will start to see results that don't match the query. Play around with the `minimum_similarity_score` to restrict the results to those with a higher relevance.

In [ ]:
embeddings = generate_embeddings("Requests realted to fire and safety")
results = execute_vector_search(embeddings, 20, 0.7)

data = []
for item in results:
    data.append([item["hotel_id"], item["hotel"], item["details"], item["SimilarityScore"]])

headers = ["Hotel Id", "Hotel", "Details", "Similarity Score"]

# Create a table to display the results
table = tabulate(data, headers, tablefmt="grid")

print(table)